# EXP-2026-010: train K1 (soft cavity targets)

GPU runtime. Run all. Send the GATE block if it stops, else the SUMMARY block.

In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"  # your fork only
BRANCH = "lumiere-onboarding"
WORK = "/content/MedSegFactory-k1-train"
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
MSF_REPO, MSF_FILE = "JohnWeck/StableDiffusion", "checkpoint-300.pth"
RUN = "K1"
RUN_TAG = f"{RUN}_r1"  # a new tag for every attempt; a run folder that holds files is refused
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")  # from MyDrive; the API follows the shortcut
OUT_BASE, OUT_RUN = ("RHUH-GBM", "synthesis"), ("EXP-2026-010", "train", RUN_TAG)


In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # before any CUDA work: repeatable cuBLAS
from google.colab import drive

drive.mount("/content/drive")  # RHUH exports and processed files are read through the mount; writes go through the API
if not Path(WORK).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
link = Path(WORK, "data", "RHUH-GBM")
link.parent.mkdir(exist_ok=True)
if not link.exists():
    link.symlink_to(DRIVE_RHUH, target_is_directory=True)
os.chdir(WORK)
# A session that ran another notebook may hold an older copy of the project in memory: use this checkout only
for key in [k for k in sys.modules if k.split(".")[0] in ("rhuh_synthesis", "rhuh_qc", "StableDiffusion")]:
    del sys.modules[key]
sys.path[:] = [p for p in sys.path if not p.startswith("/content/MedSegFactory")]
sys.path.insert(0, WORK)

from rhuh_synthesis import colab_packages  # the one package list (a module, so no name clash)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages.DATA, *colab_packages.MODEL], check=True)
commit = subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(commit)

from google.colab import auth
from googleapiclient.discovery import build

from rhuh_synthesis.drive_api import DriveRun

auth.authenticate_user()  # Colab asks once for permission to use the Drive API with this account
api = build("drive", "v3", cache_discovery=False)
settings = json.loads(Path("configs/experiments/EXP-2026-010.json").read_text(encoding="utf-8"))
LOCAL_RUN = Path(f"/content/k1_{RUN_TAG}")
assert not LOCAL_RUN.exists(), f"{LOCAL_RUN} exists: choose a new RUN_TAG or a fresh runtime"
assert RUN == settings["run"]["name"], "RUN differs from the card's run"
r2_run = DriveRun.open(api, DATA_ON_DRIVE, ("stage5_runs", "r2"), "/content/k1_r2_record",
                       expect_in_base=("split", "export_v1"))  # read only: r2's record
run_drive = DriveRun.open(api, OUT_BASE, OUT_RUN, LOCAL_RUN, expect_in_base=("EXP-2026-001",), create=True)
access = run_drive.preflight()  # a real write, update, read and removal on Drive before any GPU time
print("free GiB on Drive:", None if access["free_gib"] is None else round(access["free_gib"], 1))
assert access["free_gib"] is None or access["free_gib"] > 6, "The run needs about 6 GiB free on this account's Drive"
STARTED = time.perf_counter()


def say(message):
    print(f"[{time.perf_counter() - STARTED:6.0f} s] {message}", flush=True)


In [ ]:
import numpy as np
import torch
from diffusers import AutoencoderKL, DDPMScheduler
from huggingface_hub import hf_hub_download
from transformers import CLIPTextModel, CLIPTokenizer

from rhuh_synthesis.cavity_targets import text_config_for
from rhuh_synthesis.e_series import state_dict_sha256
from rhuh_synthesis.manifest import file_sha256
from rhuh_synthesis.model import expand_conv_in
from rhuh_synthesis.provenance import code_version, config_sha256, environment
from rhuh_synthesis.stage3 import adapter_equivalence, clip_encoder, load_reference_unet
from StableDiffusion.Our_UNet import UNet2DConditionModel

config = json.loads(Path("configs/experiments/EXP-2026-001.json").read_text(encoding="utf-8"))  # frozen, read only
S4, TRAIN = config["stage4_overfit"], config["stage5_pilot"]["training"]
text_config = text_config_for(config["text"], settings["text_class_names"])  # decision 2: ID 1 = "resection cavity"
r2 = json.loads(r2_run.fetch("record.json").read_text(encoding="utf-8"))  # by ID, checked by MD5
checkpoint = hf_hub_download(MSF_REPO, MSF_FILE)
checkpoint_sha = file_sha256(checkpoint)
assert checkpoint_sha == r2["checkpoint"]["sha256"], "The author checkpoint differs from r2's: stop and report"
unet4, load_report = load_reference_unet(UNet2DConditionModel, UNet2DConditionModel.load_config(SD_REPO, subfolder="unet"),
                                         checkpoint)
assert unet4 is not None, "Checkpoint keys do not match the vendored UNet: stop and report"
off = adapter_equivalence(unet4, latent_hw=(64, 64), cross_dim=768, device="cuda", tf32=False)
limits = S4["preflight_adapter_equivalence"]
assert off["relative_difference"] <= limits["relative_difference_max"] and off["copied_weights_identical"] and off["added_weights_zero"], \
    "Adapter preflight failed: stop and report"
unet = expand_conv_in(unet4, 11).to("cuda")
del unet4
torch.cuda.empty_cache()
step0_sha = state_dict_sha256(unet)
vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae").to("cuda").eval().requires_grad_(False)
tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder").to("cuda").eval().requires_grad_(False)
noise_scheduler = DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
encode_text = clip_encoder(tokenizer, text_encoder, "cuda")
record = {
    "experiment": "EXP-2026-010", "run": RUN, "run_tag": RUN_TAG, "commit": commit, "code_version": code_version(WORK),
    "settings_sha256": config_sha256(settings), "exp001_config_sha256": config_sha256(config),
    "environment": environment(), "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0), "train_seed": settings["run"]["train_seed"], "text_config": text_config,
    "start": {"author_checkpoint_sha256": checkpoint_sha, "equals_r2": True, "step0_state_sha256": step0_sha,
              "adapter_equivalence": off, "optimizer": "fresh AdamW"},
}
say(f"models ready; step-0 weights {step0_sha[:12]}")


In [ ]:
from rhuh_synthesis.cavity_targets import GateStats, frame_target, gate_decision, gate_patient, grid_target
from rhuh_synthesis.core_signal import reproduces_export, sequences_in_frame, visit_files
from rhuh_synthesis.e_series import (attach_lineage, check_cache, check_members, file_sha256 as sha_of_file, make_lineage,
                                     rhuh_training_members, settings_of, sha256_of)
from rhuh_synthesis.pilot import local_copies
from rhuh_synthesis.training import build_feature_cache, merge_caches, slice_pools
from rhuh_synthesis.working_grid import to_working_grid

PRIVATE = Path("data/RHUH-GBM/synthesis/EXP-2026-001")
split = json.loads((PRIVATE / "split/split.json").read_text(encoding="utf-8"))
policy = json.loads((PRIVATE / "registration_policy/acceptance_policy_v1.json").read_text(encoding="utf-8"))
manifest = json.loads((PRIVATE / "manifest/case_manifest.json").read_text(encoding="utf-8"))["patients"]
members = rhuh_training_members(split, policy)
assert len(members) == r2["data_summary"]["training_patients"], "RHUH's training set differs from r2's: stop and report"
originals = {p: next(PRIVATE.glob(f"export_v1/train*/{p}.npz")) for p in members}
paths = local_copies(originals, "/content/k1_train_exports")
cavity_cfg, gate_stats, cache, pools, target_stats = settings["cavity_target"], GateStats(), None, {}, []
for i, patient in enumerate(members, start=1):
    with np.load(paths[patient]) as data:
        arrays = {key: data[key] for key in data.files}
    sidecar = json.loads(originals[patient].with_suffix(".json").read_text(encoding="utf-8"))
    source = manifest[patient]["post"]["t1ce"]
    t1ce_file = Path(source["path"])
    assert t1ce_file.exists(), f"patient {i}: the processed post-op T1ce is not reachable on Drive"
    assert sha_of_file(t1ce_file) == source["file_sha256"] == sidecar["source_hashes"]["post"]["t1ce"], \
        f"patient {i}: the post-op T1ce file differs from the manifest or the export"
    sequences = sequences_in_frame(visit_files(t1ce_file), sidecar)
    difference, same_foreground = reproduces_export(sequences, arrays, sidecar)
    assert difference <= 1e-6 and same_foreground, f"patient {i}: the moved T1ce does not reproduce the export"
    p_frame, stats, reason = frame_target(sequences, arrays, config["evaluation"], cavity_cfg)
    assert not reason, f"patient {i}: no cavity target ({reason})"
    crop, grid = to_working_grid(arrays)
    grid["cavity_soft"] = grid_target(p_frame, crop, grid["label"])
    del sequences, p_frame
    pools[patient] = slice_pools(grid["label"])
    one = build_feature_cache({patient: grid}, vae, encode_text, text_config, batch=16, device="cuda",
                              mask_codec=config["representation"]["mask_codec"])
    own = gate_patient(gate_stats, vae, one["patients"][patient], grid["cavity_soft"], channel_index=0)
    target_stats.append({"patient": i, **stats, "grid_soft_sum": float(grid["cavity_soft"].sum()), "gate_mae": own["mae"]})
    if cache is None:
        cache = one
    else:
        cache["patients"].update(one["patients"])
        cache["text"].update(one["text"])
    say(f"patient {i}/{len(members)}: soft {stats['soft_ml']} ml, p >= 0.5 {stats['ml_at_least_half']} ml, "
        f"gate MAE {own['mae'] if own['mae'] is None else round(own['mae'], 4)}")
gate = gate_decision(gate_stats.result(), settings["gate"])
record.update(gate=gate, cavity_targets=target_stats)
print("GATE")
print(json.dumps(gate, indent=1))
assert gate["pass"], "Decision 8: the VAE round trip loses the soft cavity channel. Training is not started: send the GATE block"
cache = merge_caches(cache, None, cohort="RHUH")
rhuh = {"split_sha256": sha256_of(split), "policy_sha256": sha256_of(policy),
        "exports": {p: sha_of_file(paths[p]) for p in members}}
lineage_settings = dict(settings_of(config, sd_repo=SD_REPO, scaling_factor=vae.config.scaling_factor),
                        text=text_config, cavity_target=cavity_cfg)
lineage = make_lineage(rhuh=rhuh, lumiere=None, settings=lineage_settings)
counts = check_members(lineage, rhuh_split=split, rhuh_policy=policy, lumiere_split=None, lumiere_accepted=None)
attach_lineage(cache, lineage)
record.update(members=counts, lineage=lineage, lineage_sha256=cache["lineage_sha256"])
say(f"cache ready: {counts}, {sum(len(p['positive']) + len(p['negative']) for p in pools.values())} slices; "
    f"lineage {cache['lineage_sha256'][:12]}")


In [ ]:
from rhuh_synthesis.training import SliceSampler, train

check_cache(cache, lineage)  # again, right before training
seed = settings["run"]["train_seed"]
optimizer = torch.optim.AdamW(unet.parameters(), lr=config["training"]["learning_rate"],
                              weight_decay=config["training"]["weight_decay"])
sampler = SliceSampler(pools, seed=seed, positive_pool_probability=config["training"]["positive_pool_probability"])
rng, generator = np.random.default_rng(seed), torch.Generator().manual_seed(seed)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
log_path = LOCAL_RUN / "train_log.jsonl"
CHECKPOINTS = Path(f"/content/k1_{RUN_TAG}_checkpoints")  # full resume checkpoints, local only
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
step = train(unet, cache, sampler, noise_scheduler, optimizer, steps=TRAIN["steps"], accumulation=TRAIN["accumulation"],
             microbatch=TRAIN["microbatch_pairs"], rng=rng, generator=generator, log_path=log_path,
             checkpoint_dir=CHECKPOINTS, checkpoint_every=config["training"]["checkpoint_every"], start_step=0,
             precision=TRAIN["precision"], config_sha256=record["exp001_config_sha256"], cohort_stats=True)
assert step == settings["run"]["evaluated_checkpoint_step"] == TRAIN["steps"]
record["train_seconds"] = time.perf_counter() - start
record["peak_gib"] = torch.cuda.max_memory_allocated() / 2**30
say(f"trained to step {step} in {record['train_seconds']:.0f} s, peak {record['peak_gib']:.1f} GiB")


In [ ]:
from rhuh_synthesis.e_series import cohort_summary
from rhuh_synthesis.manifest import write_once_json

weights = LOCAL_RUN / f"unet_step{step:06d}.pt"
torch.save(unet.state_dict(), weights)
record["weights"] = {"file": weights.name, "sha256": file_sha256(weights), "step": step}
training_log = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines()]
losses = [line["loss"] for line in training_log]
record.update(training_log=training_log, cohorts=cohort_summary(training_log),
              sampler_counts=sampler.state_dict()["counts"], total_seconds=time.perf_counter() - STARTED)
path, digest = write_once_json(LOCAL_RUN / "record.json", record)
report = run_drive.sync()
say(f"record {digest[:12]} written; Drive upload: {report}")
soft = [t["soft_ml"] for t in target_stats]
half = [t["ml_at_least_half"] for t in target_stats]
shareable = {
    "commit": commit, "run": RUN, "train_seed": record["train_seed"], "gpu": record["gpu"],
    "author_checkpoint_sha256": checkpoint_sha[:12], "step0_state_sha256": step0_sha[:12],
    "weights_sha256": record["weights"]["sha256"][:12], "lineage_sha256": record["lineage_sha256"][:12],
    "members": record["members"], "gate": gate,
    "cavity_target_ml": {"soft_median": float(np.median(soft)), "soft_range": [min(soft), max(soft)],
                         "p_at_least_half_median": float(np.median(half))},
    "cohorts": record["cohorts"],
    "train_loss": {"mean_first_50": float(np.mean(losses[:50])), "mean_at_500": float(np.mean(losses[450:500])),
                   "mean_last_50": float(np.mean(losses[-50:])), "all_finite": all(l["finite"] for l in training_log)},
    "train_seconds": round(record["train_seconds"]), "total_seconds": round(record["total_seconds"]),
    "peak_gib": round(record["peak_gib"], 1),
}


In [ ]:
assert run_drive.confirm(), f"Not every file is confirmed on Drive. The originals are in {LOCAL_RUN}: keep this session open and report"
print("Every file confirmed by the Drive server: MyDrive/" + "/".join(OUT_BASE + OUT_RUN))
print("SUMMARY")
print(json.dumps(shareable, indent=1))
